In [1]:
import pandas as pd
import gc

df = pd.read_csv('../data/fines.csv', sep = ',')

In [2]:
df.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        919
Year         930
dtype: int64

In [3]:
%%timeit
df['calculated_data'] = df['Fines'] / df['Refund'] * df['Year']

88.5 μs ± 981 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
%%timeit
answers = []
for i in range(0,len(df)) :
    value =df.iloc[i]['Fines'] / df.iloc[i]['Refund'] * df.iloc[i]['Year']
    answers.append(value)

answers = pd.Series(answers)

48.3 ms ± 1.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
results = []
for index, row in df.iterrows() :
    value = row['Fines'] / row['Refund'] * row['Year']
    results.append(value)

results = pd.Series(results)

16.5 ms ± 74.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
%%timeit
calculated_data = df.apply(
    lambda row: row['Fines'] / row['Refund'] * row['Year'],
    axis=1
)

4.61 ms ± 39 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%%timeit
df[df['CarNumber'] == 'O136HO197RUS']

136 μs ± 2.44 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
df_indexed = df.set_index('CarNumber')

In [9]:
%%timeit
df_indexed.loc['O136HO197RUS']

32.3 μs ± 787 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [10]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CarNumber        930 non-null    object 
 1   Refund           930 non-null    int64  
 2   Fines            930 non-null    float64
 3   Make             930 non-null    object 
 4   Model            919 non-null    object 
 5   Year             930 non-null    int64  
 6   calculated_data  930 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 182.1 KB


In [11]:
optimized_df = df.copy()

In [12]:
float_cols = optimized_df.select_dtypes(include=['float64']).columns
optimized_df[float_cols] = optimized_df[float_cols].astype('float32')

In [13]:
int_cols = optimized_df.select_dtypes(include=['int64']).columns

for col in int_cols:
    optimized_df[col] = pd.to_numeric(
        optimized_df[col],
        downcast='integer'
    )

In [14]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CarNumber        930 non-null    object 
 1   Refund           930 non-null    int8   
 2   Fines            930 non-null    float32
 3   Make             930 non-null    object 
 4   Model            919 non-null    object 
 5   Year             930 non-null    int16  
 6   calculated_data  930 non-null    float32
dtypes: float32(2), int16(1), int8(1), object(3)
memory usage: 163.0 KB


In [15]:
object_cols = optimized_df.select_dtypes(include=['object']).columns

for col in object_cols:
    optimized_df[col] = optimized_df[col].astype('category')

In [16]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   CarNumber        930 non-null    category
 1   Refund           930 non-null    int8    
 2   Fines            930 non-null    float32 
 3   Make             930 non-null    category
 4   Model            919 non-null    category
 5   Year             930 non-null    int16   
 6   calculated_data  930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 63.2 KB


In [17]:
import gc

del df

In [18]:
gc.collect()

15

In [19]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   CarNumber        930 non-null    category
 1   Refund           930 non-null    int8    
 2   Fines            930 non-null    float32 
 3   Make             930 non-null    category
 4   Model            919 non-null    category
 5   Year             930 non-null    int16   
 6   calculated_data  930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 63.2 KB
